# 01 — Anti-UAV410: the dataset, and mask labels for it

This is the first of five notebooks. It produces the thing every later one
consumes: **512×512 single-channel thermal clips with per-frame masks and an
`exist` flag.**

| notebook | produces |
|---|---|
| **01 (this one)** | the dataset + pseudo-mask labels |
| 02 | the fine-tuned checkpoint |
| 03 | INT8 (quantization v2) |
| 04 | SAMURAI motion-aware memory |
| 05 | size-adaptive inference |

### Why Anti-UAV410

410 thermal-infrared sequences of drones, 438K hand-annotated boxes, at
**640×512 8-bit mono**. Three properties decide everything downstream:

- A **512×512 window is native pixels — no resize at all.** That is exactly the
  `crop512` input mode the Orin deployment already runs
  (`tools/run_records.py`). Training on a resized view and deploying on a
  cropped one would train the wrong thing.
- **`exist` is annotated per frame.** That is direct supervision for
  `object_score_logits` — the signal behind the failure this project hit, where
  one hard frame writes `no_obj_ptr` into the memory bank and the next seven
  frames read it back.
- **It is video.** The memory bank *is* the model; a set of still images cannot
  train or evaluate it.

### The one environment trap in this whole pipeline

EdgeTAM installs itself **as a package called `sam2`** — it is a fork of it. So
Meta's `sam2` and EdgeTAM cannot coexist in one Python environment.

This notebook therefore runs the SAM 2.1 teacher through **`transformers`**,
which has its own SAM2 implementation under a different name and does not
collide. Notebook 02 installs EdgeTAM. **The mask store on disk is the only
thing that crosses between them.**

In [ ]:
# --- GPU ---------------------------------------------------------------
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# --- Repo --------------------------------------------------------------
# Either clone it, or point REPO at a copy already on your Drive.
import os, sys
from pathlib import Path

REPO = Path("/content/sam-dedection")
if not REPO.exists():
    !git clone -q https://github.com/yigitkayabagci/sam-dedection.git {REPO}
os.chdir(REPO)
sys.path.insert(0, str(REPO))

# The dataset layer and the metrics are pure numpy, so they can be tested here
# before anything expensive is downloaded. If this fails, stop.
!python -m unittest tests.test_antiuav_dataset tests.test_accuracy tests.test_pseudo_labels 2>&1 | tail -3

In [ ]:
# --- Dependencies ------------------------------------------------------
# transformers carries its own SAM2, so Meta's `sam2` package is never
# installed here and cannot collide with EdgeTAM in notebook 02.
!pip install -q "transformers>=4.56" opencv-python-headless gdown matplotlib

## Getting the data

Anti-UAV410 is distributed from Google Drive and Baidu disk by its authors —
there is no public direct link to hardcode here, and inventing one would just
fail at runtime.

1. Open **https://github.com/HwangBo94/Anti-UAV410** and follow its download
   link (Baidu access code `a410`).
2. Put the extracted `train/`, `val/`, `test/` folders on your Drive.
3. Point `DATA` at them below.

The full set is large. `MAX_SEQUENCES` keeps the first pass small enough to get
a labelling run and a first fine-tune done in one Colab session; raise it once
the pipeline is proven end to end.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA = Path("/content/drive/MyDrive/anti-uav410")   # <- train/ val/ test/ live here
WORK = Path("/content/drive/MyDrive/edgetam-thermal")  # labels + checkpoints
MAX_SEQUENCES = 40          # None = all 410
SIZE = 512                  # the model input, and the native crop size

WORK.mkdir(parents=True, exist_ok=True)
assert (DATA / "train").is_dir(), f"no train/ under {DATA} — see the cell above"

In [ ]:
# --- What is actually in it -------------------------------------------
import numpy as np
from src.training import frame_shape, list_sequences

train = list_sequences(DATA, "train")[:MAX_SEQUENCES]
val = list_sequences(DATA, "val")[:max(MAX_SEQUENCES // 4, 1)]
height, width = frame_shape(train[0].frames[0])

frames = sum(len(s) for s in train)
visible = sum(int(s.labels.exist.sum()) for s in train)
print(f"train: {len(train)} sequences, {frames} frames, {visible} annotated "
      f"({visible / frames:.1%})")
print(f"val:   {len(val)} sequences, {sum(len(s) for s in val)} frames")
print(f"frame size: {width}x{height}  ->  a {SIZE} crop is "
      f"{'native pixels, no resize' if (width, height) >= (SIZE, SIZE) else 'upscaled'}")

In [ ]:
# --- How big are the targets? -----------------------------------------
# This decides whether notebook 05 (size-adaptive inference) is worth its
# latency: if most targets are a handful of pixels at 512, a fixed input is
# throwing the signal away before the model sees it.
sides = np.concatenate([
    np.nanmax(np.stack([s.labels.boxes[:, 2] - s.labels.boxes[:, 0],
                        s.labels.boxes[:, 3] - s.labels.boxes[:, 1]]), axis=0)
    for s in train
])
sides = sides[np.isfinite(sides)]

buckets = [(0, 8, "tiny"), (8, 16, "small"), (16, 32, "medium"), (32, 10**6, "normal")]
print(f"{len(sides)} annotated targets, longer side in source pixels:")
for lo, hi, name in buckets:
    share = np.mean((sides >= lo) & (sides < hi))
    print(f"  {name:<7} {lo:>3}-{hi if hi < 10**6 else '+':<4} px   {share:6.1%}")
print(f"  median {np.median(sides):.1f} px, p10 {np.percentile(sides, 10):.1f}, "
      f"p90 {np.percentile(sides, 90):.1f}")

In [ ]:
# --- Clips: how many stay on native pixels? ---------------------------
# A clip gets a fixed 512 window if the target's excursion fits inside one;
# otherwise the whole frame is resized instead. The window is fixed for the
# whole clip on purpose -- SAM 2's memory bank stores features in input
# coordinates, so a window that moved between frames would put every stored
# memory in a different frame of reference from the one reading it.
from src.training import sample_clips

CLIP_LEN, CLIP_STRIDE = 8, 2

clips = sample_clips(train, length=CLIP_LEN, stride=CLIP_STRIDE, size=SIZE,
                     frame_size=(width, height), jitter=32, seed=0)
native = sum(c.native for c in clips)
print(f"{len(clips)} clips of {CLIP_LEN} frames (stride {CLIP_STRIDE})")
print(f"  {native} ({native / len(clips):.1%}) on native pixels — the deployment's crop512")
print(f"  {len(clips) - native} fall back to the resized full frame — fast passes "
      "that outrun a fixed window")

## Mask labels

Anti-UAV410 gives boxes; EdgeTAM predicts masks. A large SAM 2.1 teacher is run
once, offline, box-prompted per frame, and its masks become the training target
— the same relationship EdgeTAM already has to SAM 2, applied to one domain.

Two things carry the quality:

- **Zoom.** Prompting the teacher on the full 640×512 frame asks it to segment a
  6-pixel object, and it will not. Prompting on a crop a few times the box size
  is the same model on a much easier problem.
- **Gates.** A mask is kept only if four independent checks agree: the teacher's
  own confidence, agreement with the human box, plausible area, and being one
  connected object. Frames that fail keep their `exist` supervision and fall
  back to a box-shaped loss in notebook 02.

The acceptance rate below is a **measurement, not an assumption** — and *which*
gate rejects tells you what to fix. Mostly `area` means the zoom is wrong;
mostly `teacher_iou` means the teacher is out of its depth on this imagery.

In [ ]:
from src.training.labels import Gates, Sam2Teacher

teacher = Sam2Teacher("facebook/sam2.1-hiera-large", device="cuda")
GATES = Gates(teacher_iou=0.7, box_iou=0.6, area=(0.15, 1.3), component=0.8)
ZOOM, MIN_CROP = 4.0, 128

In [ ]:
# --- Sanity check on ONE sequence before committing to all of them -----
from src.training.labels import label_sequence

probe = label_sequence(train[0], teacher, WORK / "labels" / "train",
                       gates=GATES, zoom=ZOOM, min_size=MIN_CROP,
                       frame_size=(width, height))
print(probe)
assert probe["acceptance_rate"] > 0.3, (
    "fewer than a third of frames produced a usable mask. Raise ZOOM or "
    "MIN_CROP, or loosen the gate named most often in probe['rejected'], "
    "before spending an hour on the rest.")

In [ ]:
# --- Look at them ------------------------------------------------------
import matplotlib.pyplot as plt
from src.training import load_masks, load_window

_, masks = load_masks(WORK / "labels" / "train" / train[0].name / "pseudo_masks.npz")
picked = sorted(masks)[::max(len(masks) // 6, 1)][:6]

fig, axes = plt.subplots(1, len(picked), figsize=(3 * len(picked), 3.4))
for ax, idx in zip(np.atleast_1d(axes), picked):
    box = train[0].labels.boxes[idx]
    x0, y0, w, h = (int(v) for v in (box[0] - 40, box[1] - 40,
                                     box[2] - box[0] + 80, box[3] - box[1] + 80))
    x0, y0 = max(x0, 0), max(y0, 0)
    crop = load_window(train[0].frames[idx], (x0, y0), (w, h), 128)
    m = masks[idx][y0:y0 + h, x0:x0 + w]
    import cv2
    m = cv2.resize(m.astype(np.uint8), (128, 128), interpolation=cv2.INTER_NEAREST)
    ax.imshow(crop); ax.imshow(m, alpha=0.45, cmap="autumn"); ax.set_title(f"frame {idx}")
    ax.axis("off")
plt.suptitle(f"{train[0].name}: teacher masks that passed all four gates")
plt.tight_layout(); plt.show()

In [ ]:
# --- The whole subset --------------------------------------------------
# ~1-2 s per annotated frame on an A100. Run it once; the store is reused by
# every later notebook.
from src.training.labels import summarise
from tqdm.auto import tqdm

reports = {}
for split, sequences in (("train", train), ("val", val)):
    reports[split] = [
        label_sequence(s, teacher, WORK / "labels" / split, gates=GATES,
                       zoom=ZOOM, min_size=MIN_CROP, frame_size=(width, height))
        for s in tqdm(sequences, desc=f"labelling {split}")
    ]
    print(f"\n### {split}\n{summarise(reports[split])}\n")

In [ ]:
# --- Record what was built --------------------------------------------
import json

manifest = {
    "dataset": "Anti-UAV410",
    "frame_size": [width, height],
    "model_input": SIZE,
    "clip": {"length": CLIP_LEN, "stride": CLIP_STRIDE},
    "teacher": "facebook/sam2.1-hiera-large",
    "zoom": ZOOM, "min_crop": MIN_CROP,
    "gates": {"teacher_iou": GATES.teacher_iou, "box_iou": GATES.box_iou,
              "area": list(GATES.area), "component": GATES.component},
    "sequences": {k: [r["sequence"] for r in v] for k, v in reports.items()},
    "acceptance": {k: sum(r["accepted"] for r in v) / max(sum(r["visible"] for r in v), 1)
                   for k, v in reports.items()},
}
(WORK / "manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest["acceptance"], indent=2))
print(f"\nlabels -> {WORK / 'labels'}\nmanifest -> {WORK / 'manifest.json'}")

## What you have now, and what to check

- `WORK/labels/{train,val}/<sequence>/pseudo_masks.npz` — accepted masks, RLE.
  Frames that failed a gate are simply absent; notebook 02 reads that as
  "no mask supervision here" and uses the box loss instead.
- `WORK/manifest.json` — every parameter this run used.

**Before moving on**, read the acceptance table above rather than the mean:

| what you see | what it means |
|---|---|
| acceptance > 70 %, rejects spread evenly | good, go to notebook 02 |
| most rejects are `area` | the zoom is wrong — the teacher is grabbing background, or only a fragment |
| most rejects are `teacher_iou` | SAM 2.1 is out of its depth on this imagery; try a larger `MIN_CROP` first, and if that fails the honest answer is box-only supervision |
| most rejects are `component` | thermal clutter is being segmented alongside the drone; tighten `box_iou` |

**Next:** `02_finetune_edgetam_512_thermal.ipynb`. Use a fresh runtime — it
installs EdgeTAM, which claims the `sam2` package name.